# Guía de sustentación — Parcial GEIH (mercado laboral)

Material de **estudio personal** para sustentar el notebook "parcial tannia" ante el profesor: explica la lógica de cada punto, los conceptos transversales (factor de expansión, cruces de tablas, denominadores de las tasas) e incluye ejemplos de juguete ejecutables. No es parte de la entrega.

## Concepto 1 — El factor de expansión (FEX_C18)

La GEIH no encuesta a todos los colombianos: entrevista una **muestra** de hogares y cada persona encuestada representa a muchas personas reales del país. `FEX_C18` es ese número de representación.

**Consecuencias prácticas (aprendérselas de memoria):**

* **Nada se cuenta a mano**: cada persona vale `FEX_C18` personas. Población = `FEX_C18.sum()`, NO `len(df)`.
* **El promedio correcto es ponderado**: `promedio = Σ(ingreso × FEX_C18) / Σ(FEX_C18)` — en palabras: "el ingreso total del grupo representado, dividido por cuántas personas representa el grupo".
* Una persona de Vaupés puede representar a 200 personas reales; una de Bogotá a 50. Ignorarlo es el error más común.
* Frase para el profesor: *"todas mis cifras son estimaciones poblacionales obtenidas expandiendo la muestra con el factor FEX_C18, no conteos de la muestra"*.

In [0]:
# EJEMPLO DE JUGUETE — promedio simple vs promedio ponderado
import pandas as pd

demo = pd.DataFrame({
    'persona': ['Ana',   'Beto',   'Carla',   'Diego',   'Elena'],
    'ingreso': [1_000_000, 2_000_000, 3_000_000, 4_000_000, 5_000_000],
    'FEX_C18': [10,       20,       10,       5,        1],   # a cuántas personas reales representa
})

promedio_simple = demo['ingreso'].mean()
promedio_ponderado = (demo['ingreso'] * demo['FEX_C18']).sum() / demo['FEX_C18'].sum()

print(f"Promedio simple (cada persona vale 1):        ${promedio_simple:,.0f}")
print(f"Promedio ponderado (cada persona vale factor): ${promedio_ponderado:,.0f}")
print()
print("Por qué difieren: Beto ($2M) representa a 20 personas reales y pesa 20 veces")
print("más que Elena ($5M), que representa a 1 sola. El ponderado queda cerca de los")
print("ingresos comunes y lejos del caso extremo. Así funciona todo el parcial.")

## Concepto 2 — Cómo se conectan las tablas GEIH (llaves y banderas)

Las tablas llegan separadas pero comparten tres columnas que **juntas identifican a una única persona**:

* `DIRECTORIO` — la vivienda
* `SECUENCIA_P` — el hogar dentro de la vivienda
* `ORDEN` — la persona dentro del hogar

Todos los `merge` del parcial usan las tres columnas a la vez (cruce compuesto). ¿Por qué? Un mismo `DIRECTORIO` puede tener varios hogares, y en varios hogares se repiten los `ORDEN`. Solo el trío garantiza unicidad.

**Las banderas y columnas clave que filtran cada análisis:**

| Columna | Valor | Significado |
| --- | --- | --- |
| `PET` | 1 | Persona en edad de trabajar (15 años o más) |
| `OCI` | 1 | Ocupado |
| `DSI` | 1 | Desocupado (desempleado) |
| `P3271` | 1 / 2 | Hombre / Mujer |
| `P6040` | años | Edad |
| `DPTO` | 5, 63… | Departamento (código DANE) |
| `OFICIO_C8` | 4 dígitos | Oficio (CIUO-08) |
| `INGLABO` | pesos | Ingreso laboral mensual |

Detalle que el profesor puede preguntar: `ocupados` y `desempleados` no traen sexo ni edad, por eso se heredan cruzando con `cgse` (el merge de la celda de carga del parcial). El `how='inner'` del cruce significa "conservo solo las coincidencias"; el `how='left'` de la carga inicial significa "conservo todas las filas de la izquierda y pego lo que encuentre".

## Concepto 3 — Los denominadores correctos de las tasas (la pregunta de examen)

**Tasa de Ocupación (TO) = Ocupados / PET × 100** — "de cada 100 personas en edad de trabajar, ¿cuántas tienen empleo?" En el parcial: 59.36%.

**Tasa de Desempleo (TD) = Desocupados / PEA × 100** — "de cada 100 personas que buscan trabajo activamente, ¿cuántas no lo consiguen?" En el parcial: 8.03%.

¿Por qué denominadores distintos? Porque **el desempleo solo puede ocurrir dentro de la PEA**: una estudiante de 20 años o una ama de casa no están desempleadas, están **inactivas**. Por eso un TO de 59% NO significa que el 41% esté desempleado.

**Identidades de control (responden cualquier pregunta de población):**

* Población total = PET + no-PET (menores de 15)
* PEA = Ocupados + Desocupados
* PEI = PET − PEA
* Frase: "el desempleo vive dentro de la PEA; la inactividad vive dentro de la PET pero fuera de la PEA".

In [0]:
# EJEMPLO DE JUGUETE — un país con 1.000 personas en edad de trabajar
PET = 1000
ocupados = 590
desocupados = 50

PEA = ocupados + desocupados
PEI = PET - PEA
TO = ocupados / PET * 100      # denominador: PET
TD = desocupados / PEA * 100   # denominador: PEA

print(f"TO = {TO:.1f}%   TD = {TD:.1f}%   PEI = {PEI}")
print()
print(f"Pregunta trampa: ¿el {100-TO:.0f}% restante está desempleado?")
print(f"NO: {TD:.1f}% de la PEA está desempleado y {PEI/PET*100:.0f}% de la PET está inactivo.")

## Concepto 4 — Las herramientas pandas del parcial, una por una

* **`pd.merge(..., on=[...], how=...)`** — pegar dos tablas por columnas comunes; en el parcial, heredar sexo y edad de `cgse`.
* **`groupby(...)[...].sum()`** — "agrupar filas iguales y sumar". Siempre sumamos `FEX_C18` (población) o el ingreso ponderado. `groupby(['DPTO','P3271'])` agrupa por varios niveles a la vez (doble entrada).
* **`pd.concat([a, b], axis=1)`** — pegar columnas lado a lado; así juntamos OCUPADOS y DESOCUPADOS por grupo y la PEA nace de sumarlos.
* **`.fillna(0)`** — si un grupo no tiene desempleados, su suma no existe (NaN); se vuelve 0 para poder operar.
* **`pivot`** — reorganizar una tabla larga a ancha (una fila por departamento, columnas Hombre/Mujer).
* **`.map(diccionario)`** — traducir códigos a nombres (departamentos, sexo, oficios CIUO).
* **`.apply(funcion)`** — aplicar una función fila por fila; así se crean los rangos de edad.
* **`idxmax() / idxmin()`** — devuelven la **posición** del máximo/mínimo; con ella recuperamos qué grupo de edad tuvo la mayor o menor TD.
* **`pd.to_numeric(..., errors='coerce')`** — convertir a número; lo no convertible se vuelve NaN para filtrarlo después.
* **Spark vs pandas** — las tablas Delta se leen con `spark.read` y se bajan con `.toPandas()`; con ~30 mil filas pandas es más cómodo, Spark se justifica con millones de filas.

## La lógica punto por punto (referencia rápida)

**Punto 1 — Indicadores nacionales.** Sumar pesos de PET=1 en `ft` → PET. Total de `cgse` − PET → menores de 15. Ocupados (OCI=1) + desempleados (DSI=1) en sumas ponderadas → PEA. PET − PEA → PEI. TO = ocupados/PET; TD = desocupados/PEA. Truco: "el desempleo vive dentro de la PEA".

**Punto 2 — TD por departamento y sexo.** Ocupados agrupados por (DPTO, sexo) y desempleados igual; se pegan con `concat`; PEA del grupo = suma de ambos; TD = des/PEA. El total del departamento NO es el promedio de las dos tasas: hay que volver a sumar ambos sexos y dividir (por eso el segundo `groupby`). La tabla final pivotea a columnas Hombre/Mujer.

**Punto 3 — TD por rango de edad.** Una función clasifica `P6040` en rangos [18,24), [24,30)… — el límite superior está **excluido**: la persona de 24 cumplidos ya cae en "24 a 30", así nadie se cuenta dos veces. Se aplica sobre `cgse` y se hereda a ocupados/desempleados por merge. Igual mecánica del punto 2 pero agrupando por (DPTO, rango); `idxmax`/`idxmin` extraen el grupo extremo por departamento.

**Punto 4 — Menor TD mujeres 18–25.** Filtro de sexo (P3271=2) y edad (18 ≤ edad ≤ 25, **inclusivo**, porque la pregunta del taller lo pide así — distinto de los rangos del punto 3). Sumas ponderadas por departamento, orden ascendente, 3 primeros. El 0% de Vaupés = muestra minúscula, no ausencia real de desempleo.

**Punto 5 — INGLABO en 5 departamentos.** Filtrar OCI=1, limpiar con `to_numeric`, quitar INGLABO ≤ 0 (ceros y nulos son **no-respuesta**, no ingreso cero; incluirlos bajaría el promedio). Columna ponderada = ingreso × factor; groupby por DPTO; promedio = Σ ponderado / Σ pesos.

**Punto 6 — Ingreso por departamento × sexo × edad.** El mismo motor del punto 5 con **tres dimensiones** de agrupación (60 combinaciones). Lectura clave: los hombres ganan más en casi todos los rangos y la brecha crece con la edad.

**Punto 7 — Top 10 oficios nacional.** Igual que el 5 pero agrupando por `OFICIO_C8`, orden descendente, 10 primeros. Los nombres salen del diccionario CIUO-08 oficial de DANE (PDF verificado). Advertir que 7 de los 10 tienen muestras de 1–21 personas demuestra criterio.

**Punto 8 — Brecha por sexo en un mismo oficio (2211, médicos generales).** Al fijar el oficio se elimina la **segregación ocupacional** (que hombres y mujeres trabajen en oficios distintos) y queda la diferencia "pura" por sexo. Con 24 encuestados en 4 departamentos, la conclusión honesta es "no hay patrón sistemático en esta muestra", no "las mujeres ganan más".

**Punto 9 — Quindío, oficios extremos por sexo.** Filtrar DPTO=63, agrupar por (sexo, oficio) y a cada sexo tomar su máximo y mínimo con `sort_values().head(1)`. Los extremos inferiores (recolectores de basura ~$97 mil, técnicos en control de procesos ~$150 mil) vienen de 1–2 encuestados: lo defendible es el contraste de escala, no los montos.

## Preguntas probables del profesor (con respuesta lista)

* **¿Por qué pondera? ¿Qué pasa si no pondera?** — Sin FEX_C18 describiría la muestra, no el país; las zonas con factor alto quedarían subrepresentadas. El promedio simple nacional da ~$2.17M vs $2.2M ponderado: cercanos, pero conceptualmente distintos.
* **¿Por qué la TD divide por la PEA y no por la PET?** — Porque solo quien participa activamente (trabaja o busca) puede estar desempleado; el inactivo no compite en el mercado laboral.
* **¿Por qué filtró INGLABO > 0?** — Ceros y nulos son ausencia de respuesta, no ingreso cero; incluirlos sesgaría el promedio hacia abajo. Los ingresos menores a $100 mil que SÍ se incluyen son reales: informalidad de pocas horas.
* **¿El 59.36% de TO significa 40% de desempleados?** — No: el 8.03% de la PEA desemplea y ~35% de la PET está inactiva.
* **¿Por qué en el punto 3 el rango "18 a 24" no incluye a los de 24?** — Los rangos son [inicio, fin): el 24 abre el siguiente rango para no duplicar a nadie.
* **¿Por qué Vaupés y Putumayo en los extremos?** — Putumayo refleja una crisis laboral real; Vaupés es fragilidad de muestra (población y encuestados muy pocos).
* **Su compañera obtuvo otro top 10 en el punto 7** — Ella filtró oficios con ≥30 encuestados; los promedios de los oficios compartidos coinciden exactamente. Su versión es más robusta estadísticamente; la mía responde la pregunta literal. Se pueden defender ambas.
* **¿De dónde salen los nombres de los oficios?** — Del documento oficial CIUO-08 ajustada por DANE (`CIUO_08_AC_2015_07_21.pdf`), verificados contra el texto. La adaptación colombiana difiere de la internacional (2612 = jueces; 9611 = recolectores de basura y material reciclable, no "peones de minas").
* **¿Los ingresos son mensuales o anuales?** — Mensuales, COP, del mes anterior a la encuesta. Mediana nacional $1.75M; jueces a $12.3M calza con el escalafón real.
* **¿Por qué pandas y no Spark?** — ~30 mil filas caben en memoria; Spark se justifica a escala mayor. Leer en Spark y convertir es práctica normal de exploración.

## Checklist y frases para el día de la sustentación

**Frase marco para abrir (memorizar):**

> "El notebook estima indicadores poblacionales de la GEIH de junio de 2026 usando el factor de expansión FEX_C18; todos los promedios y tasas son ponderados."

Con eso, el profesor sabe desde el segundo cero que entiendes lo esencial.

**Tres frases que resuelven cualquier punto:**

* Para cualquier población: *"es la suma de factores del grupo"*.
* Para cualquier ingreso: *"es la suma de ingreso×factor dividida por la suma de factores"*.
* Para cualquier tasa: *"la TD usa la PEA como denominador; la TO usa la PET"*.

**Admitir límites antes de que los pregunte (suma puntos):**

* "Los departamentos amazónicos tienen muestras pequeñas y sus extremos son frágiles."
* "Los montos del punto 9 provienen de 1–2 encuestados; la conclusión es el contraste de escala."
* "Siete de los diez oficios del top nacional tienen muestras menores a 30 personas."

**Orden sugerido de exposición:**

1. Concepto del factor de expansión → 2. cómo se conectan las tablas (llaves) → 3. punto 1 (definiciones PET/PEA/PEI/tasas) → 4. puntos 2–4 (desempleo por sexo, edad, departamento) → 5. puntos 5–9 (ingresos y brechas) → 6. advertencias metodológicas y cierre.